# Assignment 6: 데이터 및 모델 품질 모니터링 구현
이 과제에서는 [Amazon SageMaker model monitor](https://aws.amazon.com/sagemaker/model-monitor/)를 사용하여 실시간 추론 엔드포인트에 대한 지속적인 데이터 품질 모니터링을 구현합니다.

이 과제의 연습을 위한 코드 스니펫과 일반적인 지침은 [`06-monitoring.ipynb`](../06-monitoring.ipynb) 노트북을 참조하십시오.

## 패키지 임포트

In [ ]:
%pip install jsonlines tqdm

In [ ]:
import boto3
import botocore
import sagemaker 
import json
import jsonlines
import random
from tqdm import trange
from sagemaker.predictor import Predictor
import time
from time import gmtime, strftime
from datetime import datetime, timedelta
import uuid
import pandas as pd
import numpy as np
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    DataCaptureConfig,
    CronExpressionGenerator,
    ModelQualityMonitor,
    EndpointInput,
)
from sagemaker.model_monitor.dataset_format import DatasetFormat
from utils.monitoring_utils import run_model_monitor_job
from sagemaker.s3 import S3Downloader, S3Uploader
from sagemaker.clarify import (
    BiasConfig,
    DataConfig,
    ModelConfig,
    ModelPredictedLabelConfig,
    SHAPConfig,
)
from urllib.parse import urlparse

In [ ]:
sm = boto3.client("sagemaker")
s3 = boto3.client("s3")
session = sagemaker.Session()
pd.set_option("display.max_colwidth", None)

## 연습 1: 데이터 캡처 구성 확인
이전 노트북에서 배포한 기존 추론 엔드포인트 중 하나를 사용하십시오. 데이터 캡처는 스테이징의 경우 들어오는 데이터의 100%, 프로덕션 엔드포인트의 경우 80%로 구성되어 있습니다. Studio UX의 **Endpoint details** 보기에서 이 구성을 확인하십시오.

![](../img/endpoints.png)

![](../img/endpoint-details-data-capture.png)

`boto3`를 사용하여 엔드포인트를 설명할 수도 있습니다.

In [ ]:
# 엔드포인트의 세부 정보 가져오기
# ep_name = 
# sm.describe_endpoint()

In [ ]:
# 캡처된 데이터 파일이 저장되는 S3 URL 가져오기
# data_capture_uri = sm.describe_endpoint(EndpointName=ep_name)['DataCaptureConfig']['DestinationS3Uri']

## 연습 2: 캡처된 데이터 생성 및 확인
이 연습에서는 캡처된 데이터를 생성하기 위해 추론 엔드포인트에 데이터를 전송합니다. SageMaker Python SDK 클래스 [Predictor](https://sagemaker.readthedocs.io/en/stable/api/inference/predictors.html#sagemaker.predictor.Predictor)를 사용하여 엔드포인트와 상호 작용하십시오.

In [ ]:
# 엔드포인트 이름에서 predictor 생성
# endpoint_name = 
# predictor = Predictor()

테스트 데이터로 `02-sagemaker-containers.ipynb` 노트북에서 생성된 `tmp` 폴더의 테스트 데이터셋을 사용할 수 있습니다. 테스트 데이터셋이 없는 경우 모델 빌드 파이프라인을 실행하고 Amazon S3 버킷에서 테스트 데이터셋을 `tmp` 폴더로 다운로드하여 생성할 수 있습니다.

In [ ]:
# 테스트 데이터 로드
# test_x = pd.read_csv()

In [ ]:
# 엔드포인트에 데이터 전송
def generate_endpoint_traffic(predictor, data):
    l = len(data)
    print(f"{l}개의 벡터를 엔드포인트로 전송 중")
    for i in trange(l):
        predictions = np.array(predictor.predict(data.iloc[i].values), dtype=float).squeeze()
        time.sleep(0.001)

In [ ]:
# 엔드포인트 트래픽 생성
# generate_endpoint_traffic(predictor, test_x)

캡처된 데이터가 포함된 파일이 Amazon S3 버킷에 나타날 때까지 몇 분 기다리십시오.

각 추론 요청은 `jsonl` 파일의 한 줄에 캡처됩니다. 이 줄에는 입력과 출력이 함께 병합되어 포함됩니다.

In [ ]:
# 캡처 S3 접두사의 파일 나열
# !aws s3 ls {data_capture_uri} --recursive

In [ ]:
# 마지막으로 캡처된 데이터셋을 Studio의 EFS로 다운로드

In [ ]:
# jsonl 객체 출력

## 연습 3: 베이스라인 데이터 프로파일링 실행
데이터 모니터링을 활성화하려면 먼저 베이스라인 통계 및 제약 조건을 생성해야 합니다.

### 베이스라인 작업 생성
데이터를 프로파일링하고 베이스라인을 생성하려면 모델 빌드 파이프라인에서 생성된 베이스라인 데이터셋 `baseline.csv`를 사용하십시오. 베이스라인 데이터셋이 없는 경우 파이프라인을 실행하십시오. 베이스라인 데이터셋에 대한 S3 경로를 가져오려면 `03-assignment-sagemaker-pipeline.ipynb` 노트북을 참조하십시오.

[DefaultModelMonitor](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.DefaultModelMonitor)를 사용하여 SageMaker 모델 모니터 기능과 상호 작용하십시오. 베이스라인을 생성하려면 [`suggest_baseline`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.DefaultModelMonitor.suggest_baseline) 메서드를 호출하십시오.

In [ ]:
# 지정된 S3 URL 아래에 베이스라인 데이터셋이 있는지 확인
# !aws s3 ls {baseline_s3_url}/

In [ ]:
# 환경에 맞는 Amazon S3 경로 설정
baseline_results_s3_url = "<베이스라인 결과가 저장될 위치>"
reports_s3_url = "<모니터링 작업 보고서가 저장될 위치>"
baseline_dataset_uri = "<파일 이름을 포함한 베이스라인 데이터셋을 가리킴>"
baseline_job_name = "<SageMaker 콘솔에서 인식할 수 있는 작업 이름>"

In [ ]:
# DefaultModelMonitor 생성
# data_monitor = DefaultModelMonitor()

# 프로파일링 작업 실행
# data_monitor.suggest_baseline()

프로파일링 작업이 완료될 때까지 기다리십시오.

### 생성된 통계 및 제약 조건 확인
베이스라인 작업은 베이스라인 통계를 `statistics.json` 파일에 저장하고 제안된 베이스라인 제약 조건을 `output_s3_uri` 매개변수로 지정한 위치의 `constraints.json` 파일에 저장합니다.

`DefaultModelMonitor.latest_baselining_job` 속성의 [`baseline_statistics()`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.BaseliningJob.baseline_statistics) 및 [`suggested_constraints()`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.BaseliningJob.suggested_constraints) 메서드를 통해 통계 및 제약 조건에 액세스할 수도 있습니다.

프로파일링 작업이 생성한 통계 및 제약 조건을 탐색하십시오.

In [ ]:
# !aws s3 ls {baseline_results_s3_url}/

In [ ]:
# 베이스라인 데이터셋에 대해 생성된 제약 조건 및 통계 탐색
# baseline_job = data_monitor.latest_baselining_job

`statistics.json` 및 `constraints.json`에서 정규화된 JSON을 Pandas DataFrame으로 로드할 수도 있습니다.

In [ ]:
# statistics_df = pd.json_normalize(baseline_job.baseline_statistics().body_dict["features"])

## 연습 4: 데이터 품질 모니터링
베이스라인 제약 조건 및 통계를 생성한 후 이제 들어오는 데이터가 동일한 통계 분포를 가지고 구성된 모든 제약 조건을 준수하는지 확인할 수 있습니다.

Model Monitor 분석기의 예약된 실행을 사용하거나 분석기를 SageMaker 처리 작업으로 수동으로 실행할 수 있습니다.

Model Monitor는 구성된 [모니터링 일정](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-schedule-data-monitor.html)에 따라 주기적으로 캡처된 데이터를 베이스라인과 비교합니다.

분석기를 수동으로 실행하는 경우 베이스라인 통계 및 제약 조건을 SageMaker 처리 작업 매개변수로 제공합니다.

### 모니터링 일정 생성
모니터링 일정을 생성하려면 `DefaultModelMonitor` 클래스의 [`create_monitoring_schedule()`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.DefaultModelMonitor.create_monitoring_schedule) 메서드를 사용하십시오. cron 표현식 문자열을 생성하려면 [`CronExpressionGenerator`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.cron_expression_generator.CronExpressionGenerator) 클래스를 사용하십시오.

모니터링 베이스라인을 생성할 때 레이블이 없는 모든 기능이 포함된 베이스라인 데이터셋을 사용했습니다. Model Monitor는 기본적으로 모델의 입력과 출력을 연결하여 모든 기능과 레이블이 포함된 데이터셋을 생성합니다. Model Monitor 분석기에 전달하기 전에 레코드를 전처리하지 않으면 베이스라인 데이터셋의 열 수가 레코드의 열 수와 일치하지 않아 Model Monitor가 `extra_column_check` 위반을 보고합니다. 이러한 상황을 피하려면 베이스라인에 레이블 열을 포함하거나 모니터링되는 레코드에서 모델 출력을 제거해야 합니다. 레이블 없이 입력 데이터만 반환하는 [사용자 정의 레코드 전처리](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-pre-and-post-processing.html) 스크립트를 사용할 수 있습니다. 자세한 내용은 [`06-monitoring.ipynb`](../06-monitoring.ipynb) 노트북을 참조하십시오.

In [ ]:
# 사용자 정의 레코드 전처리 스크립트 탐색
!pygmentize ../record_preprocessor.py

In [ ]:
# 레코드 전처리 스크립트를 S3에 업로드

In [ ]:
# 모니터링 일정 이름 설정 및 모니터링 일정 생성
# mon_schedule_name = # 모니터링 일정에 대한 고유한 이름 사용
# data_monitor.create_monitoring_schedule()

In [ ]:
# 모니터링 일정 세부 정보 가져오기
## data_monitor.describe_schedule()

### 준수 트래픽 생성
`generate_endpoint_traffic` 헬퍼 함수를 사용하여 엔드포인트 트래픽을 생성하십시오.

In [ ]:
# generate_endpoint_traffic(predictor, test_x)

In [ ]:
### {data_capture_uri} 아래에 캡처된 데이터 확인

### 수동 모니터링 작업 시작
구성된 예약된 Model Monitor 실행이 시작될 때까지 기다리지 않으려면 [내장 컨테이너](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-pre-built-container.html)와 SageMaker [처리 작업](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html)을 사용하여 분석기를 수동으로 실행할 수 있습니다.

이 [리포지토리](https://github.com/aws-samples/reinvent2019-aim362-sagemaker-debugger-model-monitor/tree/master/02_deploy_and_monitor)의 소스 코드를 참조하십시오. 이 리포지토리에도 [헬퍼 함수](../utils/monitoring_utils.py)의 복사본이 있습니다.

In [ ]:
!pygmentize ../utils/monitoring_utils.py

In [ ]:
# 매개변수를 설정하고 Model Analyzer 처리 작업 실행
# utils.monitoring_utils.run_model_monitor_job()

### 모니터링 작업 출력 탐색
분석기를 SageMaker 처리 작업으로 실행하므로 표준 API를 통해 모든 작업 세부 정보에 액세스할 수 있습니다. 예를 들어 작업 출력에 대한 S3 URI를 검색할 수 있습니다.

In [ ]:
analyzer_job_name = sm.list_processing_jobs(
    NameContains = 'sagemaker-model-monitor-analyzer',
    SortOrder='Descending',
    MaxResults=2
)['ProcessingJobSummaries'][0]['ProcessingJobName']

analyzer_job_info = sm.describe_processing_job(
    ProcessingJobName=analyzer_job_name
)

analyzer_job_output_s3_url = analyzer_job_info['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']

print(analyzer_job_output_s3_url)

In [ ]:
# 생성된 분석기 출력 확인
!aws s3 ls {analyzer_job_output_s3_url}/

In [ ]:
# JSON 파일을 Pandas DataFrame으로 로드하고 생성된 통계, 제약 조건 및 위반 사항 탐색
# statistics = # 파일 로드
# constraints = # 파일 로드
# violations = pd.read_json(f"{analyzer_job_output_s3_url}/constraint_violations.json")

### 비준수 트래픽 생성
이제 실시간 추론 엔드포인트에 비준수 트래픽을 생성하고 Model Monitor 분석기를 다시 실행하십시오.

In [ ]:
# 이전 데이터 캡처 파일 제거

In [ ]:
# 요청에 비준수 데이터를 생성하거나 주입
# 비준수 데이터셋 준비

In [ ]:
# 비준수 데이터셋을 사용하여 트래픽 생성

In [ ]:
# Model Monitor 분석기 처리 작업 시작

In [ ]:
# 분석기 보고서 탐색
# 이전 섹션과 동일한 코드 사용

### 예약된 실행 및 모니터링 보고서 작업
Model Monitor 예약된 실행은 분석기 실행 및 모니터링 보고서 작업에 대한 보다 추상적인 방법을 제공합니다. SageMaker 처리 작업 API를 사용하는 대신 Python SDK [`ModelMonitor`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.ModelMonitor) 파생 클래스를 사용하여 모든 예약된 실행, 실행 세부 정보, 각 실행에 대해 생성된 통계, 제약 조건 및 위반 사항에 액세스할 수 있습니다.

예약된 실행은 마지막 Model Monitor 분석기 실행 이후 가장 최근에 캡처된 데이터만 자동으로 처리합니다. SageMaker Studio에서 [데이터 품질 보고서를 시각화](https://sagemaker-examples.readthedocs.io/en/latest/sagemaker_model_monitor/visualization/SageMaker-Model-Monitor-Visualize.html)할 수도 있습니다.

결과 해석은 [SageMaker Model Monitor 개발 가이드](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-interpreting-violations.html)를 참조하십시오.

#### 예약된 모델 모니터링 작업의 실행 나열
`ModelMonitor` Python SDK 클래스의 [`list_executions()`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.ModelMonitor.list_executions)를 사용하십시오.

In [ ]:
# 모든 실행 나열
# 최신 실행 세부 정보 가져오기
# 실행 출력 S3 URL 가져오기

#### 최신 실행 통계 및 제약 조건 가져오기
이 코드로 최신 출력에 액세스할 수 있습니다:
```
my_executions = my_monitor.list_executions()
lastest_execution_statistics = my_executions[-1].statistics()
lastest_execution_violations = my_executions[-1].constraint_violations()
```

In [ ]:
# 최신 통계 및 제약 조건 위반을 출력하는 코드 작성
# 힌트: Pandas DataFrame을 사용하여 보고서 시각화

#### 베이스라인 및 최신 데이터 프로파일링 통계 확인
[`latest_monitoring_statistics()`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.ModelMonitor.latest_monitoring_statistics) 및 [`baseline_statistics()`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.ModelMonitor.baseline_statistics) 메서드를 사용하여 모니터링 출력을 로드하십시오.

In [ ]:
# 최신 모니터링 통계를 확인하는 코드 작성

#### 위반 보고서 확인
[`latest_monitoring_constraint_violations()`](https://sagemaker.readthedocs.io/en/stable/api/inference/model_monitor.html#sagemaker.model_monitor.model_monitoring.ModelMonitor.latest_monitoring_constraint_violations)를 사용하여 최신 제약 조건 위반 보고서를 반환하십시오.

In [ ]:
# 최신 제약 조건 위반 보고서를 Pandas DataFrame으로 로드

In [ ]:
# 데이터 모니터링 결과 탐색

---

## 연습 5: 모델 품질 모니터링
모델 품질 모니터링 구현은 ground truth 데이터 수집이 추가된 데이터 품질 모니터링과 동일한 단계를 따릅니다.

모델 품질 모니터링에 대한 [Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-model-quality.html) 문서와 [step 6](../06-monitoring.ipynb) 노트북의 **Part 2: Monitor model quality**를 참조하십시오.

### 모델 품질 모니터 생성

In [ ]:
# model_monitor = ModelQualityMonitor(...)

### 모델 품질 베이스라인 작업 실행

In [ ]:
# model_baseline_job = model_monitor.suggest_baseline(...)

### 생성된 베이스라인 보고서 검사

In [ ]:
# latest_model_baseline_job = model_monitor.latest_baselining_job

### 엔드포인트 트래픽 생성

### Ground truth 데이터 수집
`EventId` 식별자를 통해 ground truth 레이블을 추론 입력과 연관시키는 것을 기억하십시오

### 모델 모니터링 일정 생성

In [ ]:
# endpoint_input = EndpointInput(...)
# model_monitor.create_monitoring_schedule(...)

### 모델 모니터 실행 및 보고서 검사

In [ ]:
# model_mon_executions = model_monitor.list_executions()

## 정리 작업 계속하기
과제와 실험을 완료한 후에는 생성된 모든 리소스를 정리해야 합니다.

[clean-up](../99-clean-up.ipynb) 노트북으로 이동하십시오.